In [ ]:
# calculate_positions.py
"""
Calculate batting positions from ball-by-ball data and save to CSV.
Run this once whenever you have new data.
"""

import pandas as pd
import numpy as np
import sqlite3


print("="*80)
print("CALCULATE BATTING POSITIONS FROM BALL-BY-BALL DATA")
print("="*80)


# Step 1: Load ball-by-ball data
print("\n Loading ball_by_ball_canonical.csv...")
bbb = pd.read_csv('C:/Desktop/crickanal/outputs/ball_by_ball_canonical.csv')

bbb.rename(columns={'batter': 'striker', 'is_wicket': 'iswicket'}, inplace=True)
print("    Renamed columns: batter → striker, is_wicket → iswicket")

print(f" Loaded {len(bbb):,} balls")
print(f"   Matches: {bbb['match_id'].nunique()}")
print(f"   Players (strikers): {bbb['striker'].nunique()}")


# Step 2: Calculate positions using pure vectorized operations
print("\n Calculating batting positions (fully vectorized)...")

# Sort by match, innings, ball
bbb_sorted = bbb.sort_values(['match_id', 'innings', 'ball']).copy()

# Calculate cumulative wickets for each ball
bbb_sorted['cumulative_wickets'] = bbb_sorted.groupby(['match_id', 'innings'])['iswicket'].cumsum()

# Shift to get wickets BEFORE current ball
bbb_sorted['wickets_before'] = bbb_sorted.groupby(['match_id', 'innings'])['cumulative_wickets'].shift(1).fillna(0)

# Get first ball for each striker in each innings
first_ball_mask = ~bbb_sorted.duplicated(subset=['match_id', 'innings', 'striker'], keep='first')
first_balls_df = bbb_sorted[first_ball_mask].copy()

# Calculate position based on wickets_before
first_balls_df['position'] = np.where(
    first_balls_df['wickets_before'] > 0,
    np.minimum(first_balls_df['wickets_before'] + 2, 11),
    0
)

# For openers
opener_mask = first_balls_df['wickets_before'] == 0
opener_df = first_balls_df[opener_mask].copy()
opener_df = opener_df.sort_values(['match_id', 'innings', 'ball'])
opener_df['opener_order'] = opener_df.groupby(['match_id', 'innings']).cumcount()
opener_df['position'] = np.where(opener_df['opener_order'] == 0, 1, 2)
first_balls_df.loc[opener_mask, 'position'] = opener_df['position'].values

print(f" Calculated positions for {len(first_balls_df):,} player-innings")


# Step 3: Aggregate by player
print("\n Aggregating by player...")

player_positions = first_balls_df.groupby('striker').agg({
    'position': ['mean', 'median', 'std', 'min', 'max', 'count']
}).reset_index()

player_positions.columns = ['player', 'avg_position', 'median_position', 'std_position', 
                            'min_position', 'max_position', 'innings_count']

# Round values
player_positions['avg_position'] = player_positions['avg_position'].round(1)
player_positions['median_position'] = player_positions['median_position'].round(1)
player_positions['std_position'] = player_positions['std_position'].round(2)

print(f" Aggregated for {len(player_positions):,} unique players")


# Step 4: Apply role-based position floor
print("\n Applying role-based position adjustments...")

db_paths = ['analytics.db', 'C:/Desktop/crickanal/analytics.db', '../analytics.db']
conn = None
roles_df = pd.DataFrame()

for db_path in db_paths:
    try:
        conn = sqlite3.connect(db_path)
        # Test if connection works
        conn.execute("SELECT 1").fetchone()
        print(f"    Connected to database: {db_path}")
        
        # get player roles
        try:
            roles_query = "SELECT DISTINCT player, role, balls_faced, balls_bowled FROM player_final_vectors"
            roles_df = pd.read_sql(roles_query, conn)
            print(f"    Using table: player_final_vectors ({len(roles_df)} players)")
            break
        except:
            try:
                roles_query = "SELECT DISTINCT player, role, balls_faced, balls_bowled FROM player_vectors"
                roles_df = pd.read_sql(roles_query, conn)
                print(f"    Using table: player_vectors ({len(roles_df)} players)")
                break
            except:
                continue
    except:
        continue

if len(roles_df) > 0:
    # Merge with positions
    player_positions = player_positions.merge(roles_df, left_on='player', right_on='player', how='left')
    
    # Apply role-based floor for pure bowlers
    # Pure bowlers (BOWL) who bat very rarely should be minimum position 8
    bowler_mask = (player_positions['role'] == 'BOWL') & \
                  (player_positions['balls_bowled'].fillna(0) > player_positions['balls_faced'].fillna(0) * 3)
    
    adjusted_count = 0
    
    # For pure bowlers, use 75th percentile or 8.0, whichever is higher
    for idx in player_positions[bowler_mask].index:
        player = player_positions.loc[idx, 'player']
        player_innings = first_balls_df[first_balls_df['striker'] == player]['position']
        
        if len(player_innings) >= 5:
            percentile_75 = player_innings.quantile(0.75)
            adjusted_pos = max(percentile_75, 8.0)  # Minimum 8 for pure bowlers

            if adjusted_pos > player_positions.loc[idx, 'avg_position']:
                player_positions.loc[idx, 'avg_position'] = round(adjusted_pos, 1)
                player_positions.loc[idx, 'median_position'] = round(player_innings.median(), 1)
                adjusted_count += 1
    
    print(f"    Adjusted positions for {adjusted_count} pure bowlers")
    
    # Drop temporary columns
    player_positions = player_positions.drop(columns=['role', 'balls_faced', 'balls_bowled'])
else:
    print("   ⚠️  No role data found - skipping role adjustment")

if conn:
    conn.close()

print(f" Final dataset: {len(player_positions):,} players")


# Step 5: Save to CSV
print("\n Saving to CSV...")

player_positions.to_csv('C:/Desktop/crickanal/outputs/player_batting_positions.csv', index=False)
print(" Saved to outputs/player_batting_positions.csv")


# Step 6: Show summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print("\nPosition Distribution:")
print(player_positions['avg_position'].describe())

print("\nTop 20 Players (Most Innings):")
top20 = player_positions.nlargest(20, 'innings_count')[['player', 'avg_position', 'median_position', 'innings_count']]
print(top20.to_string(index=False))

print("\nOpeners (Position 1-2.5):")
openers = player_positions[player_positions['avg_position'] <= 2.5]
print(f"Count: {len(openers)}")
if len(openers) > 0:
    top_openers = openers.nlargest(10, 'innings_count')[['player', 'avg_position', 'innings_count']]
    print(top_openers.to_string(index=False))

print("\nMiddle Order (Position 3-6):")
middle = player_positions[(player_positions['avg_position'] > 2.5) & (player_positions['avg_position'] <= 6.5)]
print(f"Count: {len(middle)}")
if len(middle) > 0:
    top_middle = middle.nlargest(10, 'innings_count')[['player', 'avg_position', 'innings_count']]
    print(top_middle.to_string(index=False))

print("\nTail-enders (Position 8+):")
tail = player_positions[player_positions['avg_position'] >= 8]
print(f"Count: {len(tail)}")
if len(tail) > 0:
    top_tail = tail.nlargest(10, 'innings_count')[['player', 'avg_position', 'innings_count']]
    print(top_tail.to_string(index=False))

# Show specific bowlers
print("\nKey Indian Bowlers:")
check_bowlers = ['Mohammed Shami', 'Mohammed Siraj', 'JJ Bumrah', 'Arshdeep Singh', 'Kuldeep Yadav']
for bowler in check_bowlers:
    match = player_positions[player_positions['player'] == bowler]
    if len(match) > 0:
        print(f"  {bowler}: {match.iloc[0]['avg_position']}")

print("\n" + "="*80)
print(" COMPLETE - CSV ready for database loading")
print("="*80)


CALCULATE BATTING POSITIONS FROM BALL-BY-BALL DATA

📂 Loading ball_by_ball_canonical.csv...
   ✅ Renamed columns: batter → striker, is_wicket → iswicket
✅ Loaded 1,394,613 balls
   Matches: 6055
   Players (strikers): 3537

🎯 Calculating batting positions (fully vectorized)...
✅ Calculated positions for 94,391 player-innings

📊 Aggregating by player...
✅ Aggregated for 3,537 unique players

🔧 Applying role-based position adjustments...
   ✅ Connected to database: analytics.db
   ✅ Connected to database: C:/Desktop/crickanal/analytics.db
   ✅ Using table: player_vectors (1642 players)
   ✅ Adjusted positions for 628 pure bowlers
✅ Final dataset: 3,537 players

💾 Saving to CSV...
✅ Saved to outputs/player_batting_positions.csv

SUMMARY STATISTICS

Position Distribution:
count    3537.000000
mean        4.296268
std         2.496144
min         1.000000
25%         2.500000
50%         3.200000
75%         6.000000
max        11.000000
Name: avg_position, dtype: float64

Top 20 Players (M